# Demo 2: Controlled Pendulum
## Demo 2.1: Hybrid - Unified within SysSimX Framework

### Description

The following demo implements a controlled pendulum system described in Demo 2.1. The Modelica models for the `Reference`, `AngleEncoder`, `Controller`, and `Drive` are exported as an Co-Simulation FMU using OpenModelica. The FMUs are then imported and simulated in Python using the FMPy package.

The pendulum itself again modeled as an OpenSim model, which is used via the OpenSim Python API within the Co-Simulation loop.

### Features of the `SysSimX` Framework

- The Framework supports the **configuration of FMUs within a YAML configuration file** where the path to the FMU file, inputs, and outputs can be specified
- The framework provides a **unified interface for working with co-simulation FMUs and OpenSim models**
- Both FMUs and OpenSim models follow the **CoSimComponent interface protocol**, allowing for seamless integration and interaction
- Similar methods for getting and setting variables, initialization and setup, and performing simulation steps


### Procedure

**1. Changing the working dirctory to use `SysSimX` package**

In [1]:
import os
import sys
from pathlib import Path
repo_root = Path.cwd().parent.parent
sys.path.insert(0, str(repo_root))

**2. Load the configuration for the involved FMUs**

In [2]:
from SysSimX.core.config import load_config

cfg_path = repo_root / 'SysSimX' / 'demos' / 'configs' / 'demo_hybrid.yaml'

# Check if the file exists
if not cfg_path.is_file():
    raise FileNotFoundError(f"The configuration file was not found: {cfg_path}")

cfg = load_config(str(cfg_path))

print("Configuration loaded successfully:")
for key, value in cfg.items():
    print(f"{key}: {value}")
    for subkey, subvalue in value.items():
        print(f"  {subkey}: {subvalue}")
        if isinstance(subvalue, dict):
            for subsubkey, subsubvalue in subvalue.items():
                print(f"    {subsubkey}: {subsubvalue}")

ModuleNotFoundError: No module named 'yaml'

In [3]:
t0, tf, h = cfg["step"]["t0"], cfg["step"]["tf"], cfg["step"]["h"]

print(f"Simulation start time: {t0}")
print(f"Simulation end time: {tf}")
print(f"Simulation step size: {h}")

Simulation start time: 0.0
Simulation end time: 10.0
Simulation step size: 0.001


**3. Build the FMU Co-Simulation Components**

In [4]:
from SysSimX.components.fmu_component import FMUComponent

# Build FMU components
ref = FMUComponent('ref', cfg['fmus']['Reference']['path'],
                   inputs=cfg['fmus']['Reference']['inputs'],
                   outputs=cfg['fmus']['Reference']['outputs'])
sref  = FMUComponent("sref",  cfg["fmus"]["SensorRef"]["path"],
                    inputs=cfg["fmus"]["SensorRef"]["inputs"],
                    outputs=cfg["fmus"]["SensorRef"]["outputs"])
sstate= FMUComponent("sstate",cfg["fmus"]["SensorState"]["path"],
                    inputs=cfg["fmus"]["SensorState"]["inputs"],
                    outputs=cfg["fmus"]["SensorState"]["outputs"])
pid   = FMUComponent("pid",   cfg["fmus"]["PID"]["path"],
                    inputs=cfg["fmus"]["PID"]["inputs"],
                    outputs=cfg["fmus"]["PID"]["outputs"])
drive = FMUComponent("drive", cfg["fmus"]["Drive"]["path"],
                    inputs=cfg["fmus"]["Drive"]["inputs"],
                    outputs=cfg["fmus"]["Drive"]["outputs"])

**4. Build the OpenSim Pendulum Component**

In [6]:
from SysSimX.components.opensim_pendulum import OpenSimPendulum
import numpy as np

plant = OpenSimPendulum(q0=-np.pi/4, omega0=0.0)

**5. Initialize all components**

In [7]:
for component in [ref, sref, sstate, pid, drive, plant]:
    component.initialize(t0)

**6. Run the Co-Simulation with Logger**

In [8]:
from SysSimX.core.scheduler import run_controlled_pendulum

# Logger
rows = []
def log(t, signals):
    rows.append({"t": t, **signals})

run_controlled_pendulum(ref=ref, sensor_ref=sref, sensor_state=sstate,
                        pid=pid, drive=drive, plant=plant,
                        t0=t0, tf=tf, h=h, logger=log)

print(f"Simulation completed with {len(rows)} steps.")
print("Final state values:")
for key, value in rows[-1].items():
    print(f"{key:<11}: {value:.4f}")

Simulation completed with 10000 steps.
Final state values:
t          : 9.9990
q_ref      : 0.0000
q_state    : 0.1597
omega_state: -1.1362
U_q_ref    : 1.4941
U_q_state  : 1.6588
u_pid      : -0.0284
torque     : 24.8262


**7. Plot the results**

In [13]:
import numpy as np
t_vals = [row['t'] for row in rows]
q_ref_vals = [row['q_ref'] for row in rows]
q_state_vals = [row['q_state'] for row in rows]
omega_state_vals = [row['omega_state'] for row in rows]

# Create a simple plotly figure for the q_ref and q_state over time
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_vals, y=q_ref_vals, mode='lines',
                            name='Reference Angle (q_ref)', line=dict(color='white', dash='dash')))
fig.add_trace(go.Scatter(x=t_vals, y=q_state_vals, mode='lines',
                            name='State Angle (q_state)', line=dict(color='red')))
fig.update_layout(
    title={'text': 'Pendulum Angle - FMUs + OpenSim Pendulum', 'font': {'size': 24}},
    xaxis_title={'text': 'Time (s)', 'font': {'size': 20}},
    yaxis_title={'text': 'Angle (rad)', 'font': {'size': 20}},
    legend_title={'text': 'Legend', 'font': {'size': 18}},
    font={'size': 16},
    template='plotly_dark'
)

fig.show()

**8. Export results to OpenSim MOT file**

In [14]:
from SysSimX.utilities.results_opensim import create_opensim_mot_file

data = {'q': np.array([row['q_state'] for row in rows]),
        '/jointset/head_joint/q/speed': np.array(omega_state_vals)}
time = np.array([row['t'] for row in rows])
n_time_steps = time.shape[0]
filename = 'OpenSim/Results/demo_2_5.mot'

create_opensim_mot_file(data=data, time=time, filename=filename)